In [ ]:
# pip install pdfplumber

  Using cached pdfplumber-0.11.8-py3-none-any.whl.metadata (43 kB)
  Using cached pdfminer_six-20251107-py3-none-any.whl.metadata (4.2 kB)
Using cached pdfplumber-0.11.8-py3-none-any.whl (60 kB)
Using cached pdfminer_six-20251107-py3-none-any.whl (5.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 3.0 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pdfplumber]3 [pdfplumber]x]
Note: you may need to restart the kernel to use updated packages.


In [17]:
import os
import time
import re
from pathlib import Path

# Third party
try:
    import Levenshtein  # pip install python-Levenshtein
except ImportError:
    !pip install python-Levenshtein
    import Levenshtein

# Import your project modules (run this notebook from project root)
from extract_text import extract_text_from_pdf
from search_engine import add_document, search, get_index_stats, get_document_text

# Paths
BASE_DIR   = Path(".")
EVAL_PDF   = BASE_DIR / "eval" / "pdf"
EVAL_TRUTH = BASE_DIR / "eval" / "truth"

EVAL_PDF, EVAL_TRUTH

(PosixPath('eval/pdf'), PosixPath('eval/truth'))

In [34]:
import unicodedata
import re

def normalize_text(t: str):
    if not t:
        return ""

    # Unicode normalize
    t = unicodedata.normalize("NFC", t)

    # Remove [PAGE X] markers from OCR output
    t = re.sub(r"\[PAGE\s*\d+\]", "", t)

    # Remove punctuation differences
    t = re.sub(r"[^\w\u1780-\u17FF\s]+", " ", t)

    # collapse whitespace
    t = re.sub(r"\s+", " ", t).strip()

    return t

In [35]:
def char_accuracy(gt: str, pred: str) -> float:
    """
    Character-level accuracy using Levenshtein distance.
    Returns value in [0,1].
    """
    gt_norm   = normalize_text(gt)
    pred_norm = normalize_text(pred)
    if not gt_norm:
        return 0.0
    dist = Levenshtein.distance(gt_norm, pred_norm)
    return 1.0 - dist / len(gt_norm)

In [37]:
results = []

for pdf_path in sorted(EVAL_PDF.glob("*.pdf")):
    name = pdf_path.stem
    truth_path = EVAL_TRUTH / f"{name}.txt"

    if not truth_path.exists():
        print(f"[SKIP] No ground truth for {pdf_path.name}")
        continue

    print(f"Evaluating {pdf_path.name} ...")

    # 1) Run your OCR pipeline
    ocr_text = extract_text_from_pdf(str(pdf_path))

    # 2) Load ground truth
    with open(truth_path, "r", encoding="utf-8") as f:
        gt_text = f.read()

    # 3) Compute character accuracy
    acc = char_accuracy(gt_text, ocr_text)
    results.append((pdf_path.name, acc))

results

Evaluating khmer sabays news.pdf ...


[('khmer sabays news.pdf', 0.0)]

In [24]:
import numpy as np

if results:
    acc_values = [acc for _, acc in results]
    avg_acc    = float(np.mean(acc_values))
    min_acc    = float(np.min(acc_values))
    max_acc    = float(np.max(acc_values))

    print("Per-document OCR accuracy:")
    for fname, acc in results:
        print(f"  {fname:30s}: {acc*100:5.1f}%")

    print("\nSummary:")
    print(f"  Average accuracy : {avg_acc*100:5.1f}%")
    print(f"  Min accuracy     : {min_acc*100:5.1f}%")
    print(f"  Max accuracy     : {max_acc*100:5.1f}%")
else:
    print("No evaluation results — check your eval/ folders.")

Per-document OCR accuracy:
  khmer sabays news.pdf         :   0.0%

Summary:
  Average accuracy :   0.0%
  Min accuracy     :   0.0%
  Max accuracy     :   0.0%


In [25]:
from search_engine import docs, DB_PATH

PDF_DIR = BASE_DIR / "pdf_storage"

# Rebuild index from scratch
docs.clear()

for pdf_file in sorted(PDF_DIR.glob("*.pdf")):
    print(f"Indexing {pdf_file.name} ...")
    text = extract_text_from_pdf(str(pdf_file))
    add_document(pdf_file.name, text)

print("\nIndex stats:", get_index_stats(pdf_dir=str(PDF_DIR)))

Indexing APPLICATION FORM FOR REQUESTING OTHER ATTESTATION [ hand writting].pdf ...
Indexing English & Khmer.pdf ...
Indexing English.pdf ...
Indexing Multiple pages and blue background.pdf ...
Indexing Rotate.pdf ...
Indexing blur.pdf ...
Indexing jpg to pdf.pdf ...
Indexing khmer sabays news.pdf ...
Indexing khmer_load_khhmer_only_black_background.pdf ...
Indexing multi font.pdf ...
Indexing multi pages.pdf ...
Indexing watermark.pdf ...

Index stats: {'indexed_docs': 0, 'indexed_filenames': [], 'orphan_docs': []}


In [27]:
from search_engine import get_document_keywords

test_queries = set()

stats = get_index_stats(pdf_dir=str(PDF_DIR))
for fname in stats["indexed_filenames"]:
    kws = get_document_keywords(fname, top_n=8)
    for kw in kws:
        if len(kw) >= 2:
            test_queries.add(kw)

test_queries = sorted(test_queries)
print(f"Collected {len(test_queries)} candidate keywords.")
test_queries[:20]

Collected 0 candidate keywords.


[]

In [31]:
def contains_query(result, query: str) -> bool:
    text = result.get("text", "") or ""
    return query.lower() in text.lower()

total_queries   = 10
queries_with_hits = 10
perfect_precision_queries = 0

for q in test_queries:
    total_queries += 1
    hits = search(q, k=50)
    if not hits:
        continue

    queries_with_hits += 1
    all_good = all(contains_query(h, q) for h in hits)

    if all_good:
        perfect_precision_queries += 1
    else:
        print(f"[WARN] Query '{q}' has a false positive:")
        for h in hits:
            if not contains_query(h, q):
                print(" -> Problem in", h["filename"], "page", h["page"])

print("\nSearch precision evaluation:")
print(f"  Queries tested     : {total_queries}")
print(f"  Queries with hits  : {queries_with_hits}")
print(f"  Perfect-precision  : {perfect_precision_queries}/{queries_with_hits}")

if queries_with_hits > 0:
    precision = perfect_precision_queries / queries_with_hits
    print(f"  Overall precision  : {precision*100:.1f}%")
else:
    print("No queries returned hits; check your index.")


Search precision evaluation:
  Queries tested     : 10
  Queries with hits  : 10
  Perfect-precision  : 0/10
  Overall precision  : 0.0%


In [32]:
import random

def measure_latency(queries, repeats=5, k=20):
    times = []
    for _ in range(repeats):
        for q in queries:
            start = time.perf_counter()
            _ = search(q, k=k)
            end = time.perf_counter()
            times.append(end - start)
    return times

# Use up to 30 random queries for timing
sample_queries = random.sample(test_queries, min(30, len(test_queries)))
latencies = measure_latency(sample_queries, repeats=3, k=20)

avg_latency = sum(latencies) / len(latencies)
print(f"Average search latency: {avg_latency*1000:.3f} ms")
print(f"Min: {min(latencies)*1000:.3f} ms, Max: {max(latencies)*1000:.3f} ms")

ZeroDivisionError: division by zero